In [ ]:
!git clone https://github.com/rosinality/stylegan2-pytorch.git

In [ ]:
cd stylegan2-pytorch

In [ ]:
import torch
torch.manual_seed(42)

In [ ]:
import gdown
import os

files = {
    "stylegan2-ffhq-config-f.pt": "1EM87UquaoQmk17Q8d5kYIAHqu0dkYqdT",
    "model_ir_se50.pth": "1N0MZSqPRJpLfP4mFQCS14ikrVSe8vQlL",
}

for filename, file_id in files.items():
    if not os.path.exists(filename):
        gdown.download(id=file_id, output=filename, quiet=False)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
!pip install ftfy regex tqdm ninja

In [ ]:
from model import Generator

latent_dim = 512
G_original = Generator(size=1024, style_dim=latent_dim, n_mlp=8).to(device)
state_dict = torch.load("stylegan2-ffhq-config-f.pt", map_location=device)
G_original.load_state_dict(state_dict["g_ema"])
G_original.eval()

In [ ]:
#@title Examples of image generation
Run = False # @param {type:"boolean"}
import matplotlib.pyplot as plt
if Run:
    # Number of examples to generate
    num_examples = 5# @param {type:"integer"}

    for i in range(num_examples):
        # Generation of a random latent vector
        latent_z_example = torch.randn((1, latent_dim), device=device)
        generated_img_example, _ = G_original([latent_z_example], input_is_latent=False)

        # Visualization of the generated image
        plt.imshow((generated_img_example.clip(-1,1).cpu().detach()[0].permute(1, 2, 0) + 1) / 2)
        plt.title(f"{i + 1}")
        plt.axis('off')
        plt.show()

In [ ]:
!pip install ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import clip
import torch

clip_model, _ = clip.load("ViT-B/32", device=device)

In [ ]:
#prompt_src = clip.tokenize(["a photo of a person"]).to(device)
#prompt_tgt = clip.tokenize(["an anime character"]).to(device)
prompt_src = clip.tokenize(["photo of human face"]).to(device)
prompt_tgt = clip.tokenize(["anime character with cat ears, triangular cat ears on top of head, manga style artwork"]).to(device)

In [ ]:
import copy

G_source = copy.deepcopy(G_original).eval().requires_grad_(False).to(device)
G_target = copy.deepcopy(G_original).train().requires_grad_(True).to(device)

In [ ]:
import torchvision.transforms as T
clip_preprocess = T.Compose([
    T.Resize((224, 224)),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
])

In [ ]:
z_explore = torch.randn(8, latent_dim, device=device)
w_base = G_source.style(z_explore).unsqueeze(1).repeat(1, G_source.n_latent, 1)

w_opt = w_base.detach().clone().requires_grad_(True)
optimizer_w = torch.optim.Adam([w_opt], lr=0.1)

In [ ]:
with torch.no_grad():
    emb_src, emb_tgt = clip_model.encode_text(prompt_src), clip_model.encode_text(prompt_tgt)
    delta_t = emb_tgt - emb_src
    delta_t = delta_t / delta_t.norm(dim=-1, keepdim=True)

In [ ]:
with torch.no_grad():
    emb_tgt = clip_model.encode_text(prompt_tgt)
    emb_tgt = emb_tgt / emb_tgt.norm(dim=-1, keepdim=True)

In [ ]:
for _ in range(100):
    img_explore, _ = G_source([w_opt], input_is_latent=True)
    img_clip = clip_preprocess((img_explore + 1) / 2)
    
    emb_img = clip_model.encode_image(img_clip)
    emb_img = emb_img / emb_img.norm(dim=-1, keepdim=True)
    
    loss = 1.0 - torch.cosine_similarity(emb_img, emb_tgt).mean()
    optimizer_w.zero_grad()
    loss.backward()
    optimizer_w.step()

In [ ]:
# Calculate the difference (delta W) and its L2-norm for each layer
w_delta_norm = torch.norm(w_opt - w_base, dim=(0, 2))
k = 9 # Selecting the k most relevant layers
top_k_indices = torch.topk(w_delta_norm, k).indices.tolist()
print(f"The most active layers for adaptation: {top_k_indices}")

In [ ]:
for name, param in G_target.named_parameters():
    param.requires_grad = False

    if any(f"convs.{i}" in name for i in top_k_indices) and "to_rgb" not in name and "style" not in name:
        param.requires_grad = True

trainable_params = [p for p in G_target.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(trainable_params, lr=0.002)

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

def show_progress(tensor, step, num_images=4):
    print(f"Step: {step}")

    images = (tensor[:num_images].clamp(-1.0, 1.0) + 1.0) / 2.0

    grid = make_grid(images, nrow=num_images, padding=2).cpu()

    plt.figure(figsize=(4 * num_images, 4))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis('off')
    plt.show()

In [ ]:
num_preview = 8

In [ ]:
with torch.no_grad():
    fixed_z = torch.randn(num_preview, latent_dim, device=device)
    fixed_w = G_source.style(fixed_z)

In [ ]:
with torch.no_grad():
    sample_img_src, _ = G_source([fixed_w ], input_is_latent=True)
    show_progress(sample_img_src, -1, num_images=num_preview)
    sample_img_src, _ = G_target([fixed_w], input_is_latent=True)
    show_progress(sample_img_src, -1, num_images=num_preview)

In [ ]:
iterations = 1500
batch_size = 2

for i in range(iterations):

    z = torch.randn(batch_size, latent_dim, device=device)
    with torch.no_grad():
        w = G_source.style(z)
        img_src, _ = G_source([w], input_is_latent=True)


    img_tgt, _ = G_target([w], input_is_latent=True)


    img_src_clip = clip_preprocess((img_src + 1) / 2)
    img_tgt_clip = clip_preprocess((img_tgt + 1) / 2)


    with torch.no_grad():
        emb_src = clip_model.encode_image(img_src_clip)
    emb_tgt = clip_model.encode_image(img_tgt_clip)


    delta_i = emb_tgt - emb_src
    delta_i = delta_i / delta_i.norm(dim=-1, keepdim=True)
    loss = 1.0 - (delta_i * delta_t).sum(dim=-1).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (i + 1) % 50 == 0:
        with torch.no_grad():
            sample_img, _ = G_target([fixed_w], input_is_latent=True)
            show_progress(sample_img, i + 1, num_images=num_preview)